# CH101 SPAR3D artifact run

Run this once after the successful T4 diagnostic. It records a real non-empty GLB and SHA256, registers a review-only candidate, and keeps Production, Gate B, Unity, and Android locked. The existing `07_ch101_hybrid_quality_strategies.ipynb` remains the downstream Blender refine/evaluate/strict visual QA entry point.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json, os, shutil, subprocess, sys, urllib.request

CHARACTER_CODE = 'CH101'
TOOLS_REPO = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_REF = os.environ.get('RE_CAMP_BLENDER_TOOLS_REF', 'feature/ch101-free-ai3d-autobuild')
ART_MEDIA_BASE = 'https://media.githubusercontent.com/media/siri2677/re-camp'
ART_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
SPAR3D_REPO = 'https://github.com/Stability-AI/stable-point-aware-3d.git'
SPAR3D_COMMIT = 'fdc311b16809e6a8adc2f5a3407ebb3db1a95bd1'
CONTENT_ROOT = Path('/kaggle/working')
RUN_ID = os.environ.get('RE_CAMP_SPAR3D_ARTIFACT_LABEL', datetime.now(timezone.utc).strftime('v008-%Y%m%dT%H%M%SZ'))
TOOLS_DIR = CONTENT_ROOT / 're-camp-blender'
ART_DIR = CONTENT_ROOT / 're-camp'
PROVIDER_DIR = CONTENT_ROOT / 'provider-SPAR3D'
OUTPUT_ROOT = CONTENT_ROOT / 're-camp-ai3d' / CHARACTER_CODE / f'spar3d-artifact-{RUN_ID}'
REFERENCE_DIR = OUTPUT_ROOT / 'reference-views'
PROVIDER_OUTPUT = OUTPUT_ROOT / 'provider-output'
CANDIDATE_DIR = OUTPUT_ROOT / 'candidate'
REPORT = OUTPUT_ROOT / 'spar3d-artifact-run-report.json'
REFERENCE_MANIFEST = REFERENCE_DIR / 'reference-views-manifest.json'
GATES = {'sourceStatus': 'AI_GENERATED_CANDIDATE_NOT_PRODUCTION', 'gateB': 'PENDING_HUMAN_REVIEW', 'unityInputAllowed': False, 'productionPromotionAllowed': False}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({'runtime': 'kaggle', 'runId': RUN_ID, **GATES, 'secretValuesRecorded': False})

In [ ]:
def run(command, *, cwd=None, check=True, env=None):
    command = [str(item) for item in command]
    print('RUN:', ' '.join(command))
    result = subprocess.run(command, cwd=cwd, env=env, check=False)
    if check and result.returncode:
        raise RuntimeError(f'command failed ({result.returncode})')
    return result

def load_secret(name):
    if os.environ.get(name):
        return True
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        return False
    if value:
        os.environ[name] = value
        return True
    return False

secret_names = ('HF_TOKEN', 'RE_CAMP_SPAR3D_ACCESS_ACK', 'RE_CAMP_SPAR3D_LICENSE_ACK')
loaded = [name for name in secret_names if load_secret(name)]
print({'secretNames': loaded, 'secretValuesRecorded': False})
if not (TOOLS_DIR / '.git').is_dir():
    run(['git', 'clone', '--depth', '1', '--branch', TOOLS_REF, TOOLS_REPO, TOOLS_DIR])
else:
    run(['git', '-C', TOOLS_DIR, 'fetch', 'origin', TOOLS_REF])
    run(['git', '-C', TOOLS_DIR, 'checkout', '--detach', f'origin/{TOOLS_REF}'])
tools_commit = subprocess.check_output(['git', '-C', str(TOOLS_DIR), 'rev-parse', 'HEAD'], text=True).strip()
contract_path = TOOLS_DIR / 'contracts' / 'current_roster_ai3d_pipeline_v001.json'
contract = json.loads(contract_path.read_text(encoding='utf-8'))
character = next(item for item in contract['characters'] if item['character'] == CHARACTER_CODE)
locked_paths = [character['authoritativeSource'], character['generationSource']['path']] + [item['path'] for item in character.get('auxiliaryReferences', [])]
for relative_path in dict.fromkeys(locked_paths):
    destination = ART_DIR / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f'{ART_MEDIA_BASE}/{ART_COMMIT}/{relative_path}', destination)
    if destination.suffix.lower() == '.png':
        from PIL import Image
        with Image.open(destination) as image: image.verify()
print({'toolsCommit': tools_commit, 'artCommit': ART_COMMIT, 'artFilesDownloaded': len(dict.fromkeys(locked_paths))})

In [ ]:
preflight = OUTPUT_ROOT / 'spar3d-preflight.json'
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'colab_runtime_preflight.py', '--provider', 'spar3d', '--output', preflight])
preflight_payload = json.loads(preflight.read_text(encoding='utf-8'))
print({'status': preflight_payload.get('status'), 'providerPreflight': preflight_payload.get('providerPreflight', {}), **GATES})
if preflight_payload.get('status') != 'READY_GPU_VISIBLE' or preflight_payload.get('providerPreflight', {}).get('heavyweightInstallAllowed') is not True:
    raise RuntimeError('BLOCKED_PROVIDER_PREFLIGHT: heavyweight install and inference were not started')
run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'prepare_reference_views.py', '--art-root', ART_DIR, '--output-dir', REFERENCE_DIR, '--contract', contract_path, '--character', CHARACTER_CODE])
if not (PROVIDER_DIR / '.git').is_dir(): run(['git', 'clone', '--no-checkout', SPAR3D_REPO, PROVIDER_DIR])
run(['git', '-C', PROVIDER_DIR, 'fetch', '--depth', '1', 'origin', SPAR3D_COMMIT])
run(['git', '-C', PROVIDER_DIR, 'checkout', '--detach', SPAR3D_COMMIT])
provider_commit = subprocess.check_output(['git', '-C', str(PROVIDER_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if provider_commit != SPAR3D_COMMIT: raise RuntimeError('SPAR3D_COMMIT_MISMATCH')
setup_commands = [[sys.executable, '-m', 'pip', 'install', '-q', '-U', 'setuptools==69.5.1', 'wheel'], [sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-r', PROVIDER_DIR / 'requirements.txt', 'flet==0.23.1']]
setup_steps = []
for index, command in enumerate(setup_commands, start=1):
    result = run(command, cwd=PROVIDER_DIR, check=False)
    setup_steps.append({'step': index, 'returnCode': result.returncode})
    if result.returncode: raise RuntimeError(f'SPAR3D_SETUP_FAILED at step {index}')
provider_env = os.environ.copy()
provider_env['SPAR3D_DECODER_CHUNK_SIZE'] = os.environ.get('SPAR3D_DECODER_CHUNK_SIZE', '8192')
provider_env['SPAR3D_ATTENTION_QUERY_CHUNK_SIZE'] = os.environ.get('SPAR3D_ATTENTION_QUERY_CHUNK_SIZE', '256')
run_result = run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'run_spar3d_candidate.py', '--provider-repo', PROVIDER_DIR, '--input-image', REFERENCE_DIR / 'CH101_front.png', '--output-dir', PROVIDER_OUTPUT, '--preflight', preflight, '--output-report', REPORT, '--texture-resolution', os.environ.get('RE_CAMP_SPAR3D_TEXTURE_RESOLUTION', '512'), '--target-count', '20000', '--execute'], check=False, env=provider_env)
if not REPORT.is_file(): raise FileNotFoundError(REPORT)
run_payload = json.loads(REPORT.read_text(encoding='utf-8'))
run_payload.update({'toolsCommit': tools_commit, 'artCommit': ART_COMMIT, 'providerCommitExpected': SPAR3D_COMMIT, 'providerCommitActual': provider_commit, 'referenceManifest': str(REFERENCE_MANIFEST), 'setupSteps': setup_steps, **GATES})
REPORT.write_text(json.dumps(run_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print({'status': run_payload.get('status'), 'returnCode': run_payload.get('returnCode'), 'meshOutputs': run_payload.get('meshOutputs', []), 'meshSha256': run_payload.get('meshSha256', 'NOT_RECORDED'), **GATES})
if run_payload.get('status') != 'SPAR3D_EXECUTED': raise RuntimeError(f"SPAR3D artifact run did not complete: {run_payload.get('status')}")

In [ ]:
candidate_manifest = CANDIDATE_DIR / 'candidate-manifest.json'
registration = run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'register_review_candidate.py', '--mesh', run_payload['meshOutputs'][0], '--reference-manifest', REFERENCE_MANIFEST, '--output-dir', CANDIDATE_DIR, '--provider', 'spar3d', '--strategy-id', 'SPAR3D_SINGLE_VIEW_V001', '--source-stage', 'SPAR3D_T4_ARTIFACT_RESEARCH', '--candidate-label', '001', '--metadata-json', REPORT, '--contract', contract_path, '--character', CHARACTER_CODE], check=False)
if registration.returncode != 0 or not candidate_manifest.is_file(): raise RuntimeError('SPAR3D candidate registration failed')
artifact_record = OUTPUT_ROOT / 'spar3d-artifact-handoff.json'
record_result = run([sys.executable, TOOLS_DIR / 'scripts' / 'ai3d' / 'record_spar3d_artifact.py', '--run-report', REPORT, '--reference-manifest', REFERENCE_MANIFEST, '--candidate-manifest', candidate_manifest, '--output', artifact_record, '--tools-commit', tools_commit, '--art-commit', ART_COMMIT, '--contract', contract_path, '--character', CHARACTER_CODE], check=False)
if record_result.returncode != 0 or not artifact_record.is_file(): raise RuntimeError('SPAR3D artifact handoff validation failed')
artifact_payload = json.loads(artifact_record.read_text(encoding='utf-8'))
# SPAR3D_ARTIFACT_READY_FOR_REVIEW is a hash-checked artifact state, not a Production approval.
print({'artifactStatus': artifact_payload['status'], 'sourceArtifact': artifact_payload['sourceArtifact'], 'candidateManifestSha256': artifact_payload['candidateManifestSha256'], 'next': 'run 07_ch101_hybrid_quality_strategies.ipynb for Blender refine/evaluate/strict visual QA', **GATES})